# DepthPro — Unified Depth Evaluation Pipeline  *(Pre-Masked Edition)*

Evaluates **Apple DepthPro** (`apple/DepthPro-hf`) on the **KITTI Eigen-Split** dataset.

**Key change vs. original:** Binary instance masks are **loaded from the pre-generated
mask store** produced by `inst_seg_notebook.ipynb` — SAM3 is never called at runtime.

### Pipeline
| Step | What happens |
|------|-------------|
| 1 | Index KITTI Eigen-Split (test + optional train) |
| 2 | Load DepthPro (`apple/DepthPro-hf`) depth model |
| 3 | For each frame: load pre-generated masks → run DepthPro depth inference → compute scale variants |
| 4 | Write per-frame metrics to FRAME CSV and per-mask metrics to MASK CSV |
| 5 | Generate 12 analysis plots |

**Improvements over v1:**
- Dual CSV output: frame-level (`FRAME_CSV_PATH`) + mask-level (`MASK_CSV_PATH`)
- Pixel-weighted True RMSE via `true_rmse()` — replaces simple mean RMSE everywhere
- `flatten_scale_results()` helper eliminates repetitive CSV row assembly
- `n_pts` column in both CSVs for proper pixel-weighted aggregation
- `print_summary()` prints three sections: Frame-Level, Overall Mask, Per-Category
- `plot_summary_table()` renders all three evaluation tables in one figure
- Cleaner `compute_depth_errors()` — no unnecessary epsilon in GT denominators
- Better 3D error surface axis buffering
- Per-category RMSE uses pixel-weighted True RMSE

**Mask store layout** (mirrors KITTI root, output of `inst_seg_notebook.ipynb`):
```
MASK_ROOT / <split> / <date> / <drive> / image_02/data / <frame_stem> /
    <frame_stem>_manifest.json
    <frame_stem>_<super_cat>_<sub_cat>_<idx>_<uid8>_<score>_mask.png
```

**DepthPro model reference:** https://github.com/apple/ml-depth-pro

## 0 · GPU Selection & Imports

In [ ]:
# # ── GPU selection ────────────────────────────────────────────────────────────
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"   # ← edit as needed

# ── Standard library ──────────────────────────────────────────────────────────
import os
import sys
import glob
import json
import math
import logging
import warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# ── Third-party ───────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D          # noqa: F401
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

# ── DepthPro (Apple via HuggingFace) ──────────────────────────────────────────
# pip install transformers torch pillow
from transformers import DepthProImageProcessorFast, DepthProForDepthEstimation

# NOTE: SAM3 / seg_code imports removed — masks are loaded from disk.
print("Imports OK  |  CUDA available:", torch.cuda.is_available())

Imports OK  |  CUDA available: True


## 1 · Configuration  *(edit paths here)*

In [2]:
DEVICE    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CACHE_DIR = "/home/arka/Downloads/MMDE/cache"

# ── Dataset ───────────────────────────────────────────────────────────────────
KITTI_ROOT = "/home/arka/Downloads/MMDE/datasets/KITTI/complete"
TEST_ROOT  = os.path.join(KITTI_ROOT, "test")
TRAIN_ROOT = os.path.join(KITTI_ROOT, "train")
USE_TRAIN  = False        # set True to also process the training split

# ── Pre-generated mask store ──────────────────────────────────────────────────
# Root produced by inst_seg_notebook.ipynb.
# Directory tree under MASK_ROOT mirrors KITTI_ROOT exactly:
#   MASK_ROOT / <split> / <date> / <drive> / image_02/data / <frame_stem> /
MASK_ROOT = "/home/arka/Downloads/MMDE/outputs/inst_seg_output"

# ── Depth clipping ────────────────────────────────────────────────────────────
MIN_DEPTH = 1e-3
MAX_DEPTH = 80.0

# ── Camera (KITTI default) ────────────────────────────────────────────────────
FOCAL_LENGTH_X = 721.5577

# ── Mask confidence gate ──────────────────────────────────────────────────────
MIN_MASK_SCORE = 0.40

# ── DepthPro model checkpoint ─────────────────────────────────────────────────
DEPTHPRO_CHECKPOINT = "apple/DepthPro-hf"

# ── Output directories ────────────────────────────────────────────────────────
MAIN_OUTPUT_DIR = "/home/arka/Downloads/MMDE/outputs/model_analysis_output/depthpro_premasked_outputs_FINAL"
FRAME_CSV_PATH  = os.path.join(MAIN_OUTPUT_DIR, "depthpro_frame_results.csv")
MASK_CSV_PATH   = os.path.join(MAIN_OUTPUT_DIR, "depthpro_mask_results.csv")
PLOT_DIR        = os.path.join(MAIN_OUTPUT_DIR, "plots")
SAVE_NPY        = True    # save raw depth / eval-mask arrays as .npy?

# ── Resume support ────────────────────────────────────────────────────────────
START_INDEX = 0           # set > 0 to resume from a checkpoint

print("Configuration loaded.")

Configuration loaded.


## 2 · Logging

In [3]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("depthpro_premasked")
logging.getLogger("transformers").setLevel(logging.ERROR)

## 3 · Metric & Crop Helpers

In [4]:
def compute_depth_errors(gt: np.ndarray,
                         pred: np.ndarray,
                         mask: np.ndarray) -> dict:
    """Standard KITTI / Monodepth2 depth metrics (masked pixels only).

    gt and pred pixels under `mask` are assumed to be already in the valid
    depth range (MIN_DEPTH, MAX_DEPTH), so no epsilon guard is needed in the
    GT denominator.
    """
    gt_v   = gt[mask]
    pred_v = pred[mask]
    pred_v = np.maximum(pred_v, 1e-8)

    thresh   = np.maximum(gt_v / pred_v, pred_v / gt_v)
    rmse     = float(np.sqrt(np.mean((gt_v - pred_v) ** 2)))
    abs_rel  = float(np.mean(np.abs(gt_v - pred_v) / gt_v))
    sq_rel   = float(np.mean(((gt_v - pred_v) ** 2) / gt_v))
    rmse_log = float(np.sqrt(np.mean((np.log(gt_v) - np.log(pred_v)) ** 2)))
    delta1   = float(np.mean(thresh < 1.25))
    delta2   = float(np.mean(thresh < 1.25 ** 2))
    delta3   = float(np.mean(thresh < 1.25 ** 3))
    return dict(rmse=rmse, abs_rel=abs_rel, sq_rel=sq_rel,
                rmse_log=rmse_log, delta1=delta1, delta2=delta2, delta3=delta3)


def eigen_crop_mask(shape) -> np.ndarray:
    """Eigen-crop region used in standard KITTI depth benchmarking."""
    h, w = shape[:2]
    m    = np.zeros((h, w), dtype=bool)
    m[44:min(44 + 352, h), :] = True
    return m


def normalize_depth_for_display(depth: np.ndarray,
                                  cmap=plt.cm.magma) -> np.ndarray:
    """Convert a float depth map to an RGBA uint8 image for visualisation."""
    valid = depth > 0
    if not np.any(valid):
        return np.zeros((*depth.shape, 4), dtype=np.uint8)
    vmin = float(np.percentile(depth[valid], 2))
    vmax = float(np.percentile(depth[valid], 98))
    norm = np.clip((depth - vmin) / max(vmax - vmin, 1e-8), 0.0, 1.0)
    return (cmap(norm) * 255).astype(np.uint8)

print("Metric helpers defined.")

Metric helpers defined.


## 4 · Scale Computation

In [5]:
def compute_all_scales(gt_depth: np.ndarray,
                       pred_raw: np.ndarray,
                       mask: np.ndarray) -> dict:
    """
    Compute four scale-aligned depth maps from the same raw prediction.

    Returns dict keyed by 'unscaled', 'median', 'mean', 'affine'.
    Each value is a dict: scale, shift, depth_map, avg_depth, min_depth,
    plus all depth-error metrics from compute_depth_errors().
    """
    gt_v    = gt_depth[mask]
    pred_c  = np.clip(pred_raw, MIN_DEPTH, MAX_DEPTH)
    results = {}

    # ── Unscaled ─────────────────────────────────────────────────────────────
    m0 = compute_depth_errors(gt_depth, pred_c, mask)
    results["unscaled"] = {
        "scale": 1.0, "shift": 0.0, "depth_map": pred_c, **m0,
        "avg_depth": float(np.mean(pred_c[mask])),
        "min_depth": float(np.min(pred_c[mask])),
    }

    # ── Median Scale ──────────────────────────────────────────────────────────
    scale_med = float(np.median(gt_v) / (np.median(pred_c[mask]) + 1e-9))
    pred_med  = np.clip(pred_c * scale_med, MIN_DEPTH, MAX_DEPTH)
    m1 = compute_depth_errors(gt_depth, pred_med, mask)
    results["median"] = {
        "scale": scale_med, "shift": 0.0, "depth_map": pred_med, **m1,
        "avg_depth": float(np.mean(pred_med[mask])),
        "min_depth": float(np.min(pred_med[mask])),
    }

    # ── Mean Scale ────────────────────────────────────────────────────────────
    scale_mean = float(np.mean(gt_v) / (np.mean(pred_c[mask]) + 1e-9))
    pred_mean  = np.clip(pred_c * scale_mean, MIN_DEPTH, MAX_DEPTH)
    m2 = compute_depth_errors(gt_depth, pred_mean, mask)
    results["mean"] = {
        "scale": scale_mean, "shift": 0.0, "depth_map": pred_mean, **m2,
        "avg_depth": float(np.mean(pred_mean[mask])),
        "min_depth": float(np.min(pred_mean[mask])),
    }

    # ── Affine Scale  (OLS: scale * pred + shift ≈ GT) ───────────────────────
    pred_flat        = pred_c[mask].ravel()
    A                = np.vstack([pred_flat, np.ones_like(pred_flat)]).T
    params, _, _, _  = np.linalg.lstsq(A, gt_v.ravel(), rcond=None)
    scale_aff        = float(params[0])
    shift_aff        = float(params[1])
    if not np.isfinite(scale_aff) or scale_aff <= 0:
        scale_aff = scale_med
        shift_aff = 0.0
    pred_aff = np.clip(scale_aff * pred_c + shift_aff, MIN_DEPTH, MAX_DEPTH)
    m3 = compute_depth_errors(gt_depth, pred_aff, mask)
    results["affine"] = {
        "scale": scale_aff, "shift": shift_aff, "depth_map": pred_aff, **m3,
        "avg_depth": float(np.mean(pred_aff[mask])),
        "min_depth": float(np.min(pred_aff[mask])),
    }

    return results

print("Scale computation functions defined.")

Scale computation functions defined.


## 5 · DepthPro Model Loading & Inference

Uses Apple DepthPro via HuggingFace: https://github.com/apple/ml-depth-pro
Checkpoint: `apple/DepthPro-hf`

In [6]:
def load_depthpro():
    """Load DepthPro (apple/DepthPro-hf) and return (processor, model)."""
    log.info("Loading DepthPro (%s) …", DEPTHPRO_CHECKPOINT)
    proc = DepthProImageProcessorFast.from_pretrained(
        DEPTHPRO_CHECKPOINT, cache_dir=CACHE_DIR)
    model = DepthProForDepthEstimation.from_pretrained(
        DEPTHPRO_CHECKPOINT, cache_dir=CACHE_DIR
    ).to(DEVICE).eval()
    log.info("DepthPro loaded on %s.", DEVICE)
    return proc, model


def run_depthpro(im_np: np.ndarray, proc, model) -> np.ndarray:
    """
    Run DepthPro on one RGB image (uint8, H×W×3).
    Returns a float32 metric-depth map (H, W) in metres.
    """
    im     = Image.fromarray(im_np)
    inputs = proc(images=im, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outs = model(**inputs)
    post = proc.post_process_depth_estimation(
        outs, target_sizes=[im_np.shape[:2]])[0]
    return post["predicted_depth"].cpu().numpy().astype(np.float32)

print("DepthPro model helpers defined.")

DepthPro model helpers defined.


## 6 · Pre-Generated Mask Loading  *(replaces SAM3 inference)*

The two functions below resolve the mask directory for any KITTI frame and
load all pre-generated instance masks from it.  The manifest JSON is used
when available (fast path); otherwise the file names are parsed directly.

In [7]:
def get_mask_dir(rgb_path: str) -> str:
    """
    Derive the pre-generated mask directory for a given RGB image path.

    Mapping:
        KITTI_ROOT / <split> / <date> / <drive> / image_02/data / <frame>.png
           →
        MASK_ROOT  / <split> / <date> / <drive> / image_02/data / <frame_stem> /
    """
    frame_stem   = os.path.splitext(os.path.basename(rgb_path))[0]
    rel_data_dir = os.path.relpath(os.path.dirname(rgb_path), KITTI_ROOT)
    return os.path.join(MASK_ROOT, rel_data_dir, frame_stem)


def load_masks_for_frame(mask_dir: str,
                          super_cat_filter: str  = None,
                          sub_cat_filter:   str  = None,
                          min_score:        float = 0.0) -> list:
    """
    Load all pre-generated instance masks for one frame.

    Prefers the JSON manifest (fast path).  Falls back to glob + filename
    parsing when the manifest is absent.

    Parameters
    ----------
    mask_dir         : Absolute path to the frame-level mask directory.
    super_cat_filter : If given, keep only this super-category.
    sub_cat_filter   : If given, keep only this sub-category.
    min_score        : Discard instances with score < min_score.

    Returns
    -------
    List of dicts each containing:
        uid, local_idx, super_cat, sub_cat, score, box,
        mask (H×W bool ndarray), path (Path object).
    Returns an empty list if the directory does not exist or has no masks.
    """
    mask_dir_path = Path(mask_dir)
    if not mask_dir_path.exists():
        return []

    frame_stem    = mask_dir_path.name
    manifest_path = mask_dir_path / f"{frame_stem}_manifest.json"

    # ── Fast path: use manifest ───────────────────────────────────────────────
    if manifest_path.exists():
        try:
            with open(manifest_path, "r") as fh:
                records = json.load(fh)
        except (json.JSONDecodeError, OSError) as e:
            log.warning("Cannot read manifest %s: %s — falling back to glob.",
                        manifest_path, e)
            records = None

        if records is not None:
            instances = []
            for rec in records:
                if super_cat_filter and rec.get("super_cat") != super_cat_filter:
                    continue
                if sub_cat_filter   and rec.get("sub_cat")   != sub_cat_filter:
                    continue
                score = float(rec.get("score", 0.0))
                if score < min_score:
                    continue
                mask_file = mask_dir_path / rec["mask_file"]
                if not mask_file.exists():
                    log.warning("Mask file missing: %s", mask_file)
                    continue
                mask_np = np.array(Image.open(mask_file).convert("L")) > 127
                instances.append({
                    "uid":       rec.get("uid", ""),
                    "local_idx": int(rec.get("local_idx", 0)),
                    "super_cat": rec.get("super_cat", ""),
                    "sub_cat":   rec.get("sub_cat", ""),
                    "score":     score,
                    "box":       rec.get("box"),
                    "mask":      mask_np,
                    "path":      mask_file,
                })
            return instances

    # ── Fallback: glob + filename parsing ─────────────────────────────────────
    # Filename format (inst_seg_notebook.ipynb):
    #   <frame_stem>_<super_cat>_<sub_cat>_<local_idx:04d>_<uid8>_<score:.3f>_mask.png
    instances = []
    for mask_file in sorted(mask_dir_path.glob("*_mask.png")):
        stem = mask_file.stem
        if not stem.endswith("_mask"):
            continue
        stem = stem[:-5]          # strip "_mask"
        if not stem.startswith(frame_stem + "_"):
            continue
        suffix = stem[len(frame_stem) + 1:]

        parts = suffix.rsplit("_", 3)   # [categories, local_idx, uid8, score]
        if len(parts) != 4:
            log.warning("Unexpected mask filename format: %s", mask_file.name)
            continue
        cat_str, idx_str, uid_str, score_str = parts

        cat_parts = cat_str.split("_", 1)
        if len(cat_parts) != 2:
            log.warning("Cannot parse categories from: %s", cat_str)
            continue
        super_cat, sub_cat = cat_parts

        if super_cat_filter and super_cat != super_cat_filter:
            continue
        if sub_cat_filter   and sub_cat   != sub_cat_filter:
            continue

        try:
            local_idx = int(idx_str)
            score     = float(score_str)
        except ValueError:
            log.warning("Cannot parse idx/score from: %s", mask_file.name)
            continue

        if score < min_score:
            continue

        mask_np = np.array(Image.open(mask_file).convert("L")) > 127
        instances.append({
            "uid":       uid_str,
            "local_idx": local_idx,
            "super_cat": super_cat,
            "sub_cat":   sub_cat,
            "score":     score,
            "box":       None,
            "mask":      mask_np,
            "path":      mask_file,
        })

    instances.sort(key=lambda x: x["local_idx"])
    return instances

print("Mask loading helpers defined.")

Mask loading helpers defined.


## 7 · KITTI Dataset Indexing

In [8]:
def index_kitti(use_train: bool = USE_TRAIN) -> pd.DataFrame:
    """
    Walk the KITTI Eigen-Split directory and build a DataFrame of
    (RGB path, GT depth path) pairs for test (and optionally train) frames.
    """
    rows = []

    def _collect(split_root: str, tag: str) -> None:
        if not os.path.isdir(split_root):
            log.warning("Split root does not exist: %s", split_root)
            return
        pattern  = os.path.join(
            split_root, "*", "*_sync",
            "proj_depth", "groundtruth", "image_02", "*.png"
        )
        gt_paths = glob.glob(pattern)
        for gt_path in tqdm(gt_paths, desc=f"Indexing {tag}"):
            parts    = gt_path.split(os.sep)
            date, drive, fname = parts[-6], parts[-5], parts[-1]
            rgb_path = os.path.join(split_root, date, drive,
                                    "image_02", "data", fname)
            if not os.path.exists(rgb_path):
                continue
            rows.append(dict(
                folder=tag, sequence=date, drive=drive,
                filename=fname, gt_path=gt_path, rgb_path=rgb_path,
            ))

    _collect(TEST_ROOT, "Test")
    if use_train:
        _collect(TRAIN_ROOT, "Train")

    df = pd.DataFrame(rows).reset_index(drop=True)
    log.info("Dataset indexed: %d frames, %d drives.",
             len(df), df["drive"].nunique() if len(df) else 0)
    return df

print("KITTI indexing function defined.")

KITTI indexing function defined.


## 8 · CSV Column Schemas & Flatten Helper

Two CSVs are produced per run:

| CSV | Granularity | Description |
|-----|-------------|-------------|
| `FRAME_CSV_PATH` | One row per frame | Evaluated over **all** valid LiDAR pixels in the Eigen crop, independent of object masks |
| `MASK_CSV_PATH`  | One row per instance mask | Evaluated over pixels inside each specific object region |

`n_pts` stores the pixel count for each row, enabling pixel-weighted aggregation via `true_rmse()`.

In [9]:
BASE_METRIC_COLS = [
    "unscaled_rmse",      "unscaled_abs_rel",   "unscaled_sq_rel",
    "unscaled_rmse_log",  "unscaled_delta1",    "unscaled_delta2",   "unscaled_delta3",
    "unscaled_avg_depth", "unscaled_min_depth",
    "median_scale",
    "median_rmse",        "median_abs_rel",     "median_sq_rel",
    "median_rmse_log",    "median_delta1",      "median_delta2",     "median_delta3",
    "median_avg_depth",   "median_min_depth",
    "mean_scale",
    "mean_rmse",          "mean_abs_rel",       "mean_sq_rel",
    "mean_rmse_log",      "mean_delta1",        "mean_delta2",       "mean_delta3",
    "mean_avg_depth",     "mean_min_depth",
    "affine_scale",       "affine_shift",
    "affine_rmse",        "affine_abs_rel",     "affine_sq_rel",
    "affine_rmse_log",    "affine_delta1",      "affine_delta2",     "affine_delta3",
    "affine_avg_depth",   "affine_min_depth",
    "lidar_avg_depth",    "lidar_min_depth",
]

# One row per frame — evaluated over ALL valid LiDAR pixels in the Eigen crop
FRAME_CSV_COLUMNS = [
    "img_id", "file_name", "n_pts",
] + BASE_METRIC_COLS + [
    "rgb_image_path", "gt_depth_map_path", "depthpro_pred_path",
]

# One row per object instance mask
MASK_CSV_COLUMNS = [
    "img_id", "file_name",
    "mask_id", "mask_uid", "mask_score",
    "super_cat", "sub_cat", "horizontal_angle", "n_pts",
] + BASE_METRIC_COLS + [
    "rgb_image_path", "gt_depth_map_path", "depthpro_pred_path", "mask_path",
]


def flatten_scale_results(r: dict, lidar_avg: float, lidar_min: float) -> dict:
    """Un-nest compute_all_scales() output into a flat dict of CSV columns."""
    return {
        "unscaled_rmse":      r["unscaled"]["rmse"],
        "unscaled_abs_rel":   r["unscaled"]["abs_rel"],
        "unscaled_sq_rel":    r["unscaled"]["sq_rel"],
        "unscaled_rmse_log":  r["unscaled"]["rmse_log"],
        "unscaled_delta1":    r["unscaled"]["delta1"],
        "unscaled_delta2":    r["unscaled"]["delta2"],
        "unscaled_delta3":    r["unscaled"]["delta3"],
        "unscaled_avg_depth": r["unscaled"]["avg_depth"],
        "unscaled_min_depth": r["unscaled"]["min_depth"],
        "median_scale":       r["median"]["scale"],
        "median_rmse":        r["median"]["rmse"],
        "median_abs_rel":     r["median"]["abs_rel"],
        "median_sq_rel":      r["median"]["sq_rel"],
        "median_rmse_log":    r["median"]["rmse_log"],
        "median_delta1":      r["median"]["delta1"],
        "median_delta2":      r["median"]["delta2"],
        "median_delta3":      r["median"]["delta3"],
        "median_avg_depth":   r["median"]["avg_depth"],
        "median_min_depth":   r["median"]["min_depth"],
        "mean_scale":         r["mean"]["scale"],
        "mean_rmse":          r["mean"]["rmse"],
        "mean_abs_rel":       r["mean"]["abs_rel"],
        "mean_sq_rel":        r["mean"]["sq_rel"],
        "mean_rmse_log":      r["mean"]["rmse_log"],
        "mean_delta1":        r["mean"]["delta1"],
        "mean_delta2":        r["mean"]["delta2"],
        "mean_delta3":        r["mean"]["delta3"],
        "mean_avg_depth":     r["mean"]["avg_depth"],
        "mean_min_depth":     r["mean"]["min_depth"],
        "affine_scale":       r["affine"]["scale"],
        "affine_shift":       r["affine"]["shift"],
        "affine_rmse":        r["affine"]["rmse"],
        "affine_abs_rel":     r["affine"]["abs_rel"],
        "affine_sq_rel":      r["affine"]["sq_rel"],
        "affine_rmse_log":    r["affine"]["rmse_log"],
        "affine_delta1":      r["affine"]["delta1"],
        "affine_delta2":      r["affine"]["delta2"],
        "affine_delta3":      r["affine"]["delta3"],
        "affine_avg_depth":   r["affine"]["avg_depth"],
        "affine_min_depth":   r["affine"]["min_depth"],
        "lidar_avg_depth":    lidar_avg,
        "lidar_min_depth":    lidar_min,
    }


print(f"Schemas defined.  Frame cols: {len(FRAME_CSV_COLUMNS)}, Mask cols: {len(MASK_CSV_COLUMNS)}.")

Schemas defined.  Frame cols: 48, Mask cols: 55.


## 9 · Main Processing Loop

In [10]:
def process_dataset(dp_proc, dp_model, val_list: pd.DataFrame) -> None:
    """
    Iterate over every frame in val_list.  For each frame that has at least
    one pre-generated mask:

      1. FRAME-LEVEL: evaluate DepthPro prediction vs all valid LiDAR pixels
         inside the Eigen crop — completely independent of object masks.
         Writes one row to FRAME_CSV_PATH.

      2. MASK-LEVEL: for each instance mask, evaluate DepthPro prediction vs
         the LiDAR pixels that fall inside that specific object region.
         Writes one row per mask to MASK_CSV_PATH.
    """
    # ── Create output directories ─────────────────────────────────────────────
    os.makedirs(MAIN_OUTPUT_DIR, exist_ok=True)
    os.makedirs(PLOT_DIR, exist_ok=True)
    RGB_DIR  = os.path.join(MAIN_OUTPUT_DIR, "rgb_images");   os.makedirs(RGB_DIR,  exist_ok=True)
    GT_DIR   = os.path.join(MAIN_OUTPUT_DIR, "gt_depth");     os.makedirs(GT_DIR,   exist_ok=True)
    PRED_DIR = os.path.join(MAIN_OUTPUT_DIR, "depthpro_pred"); os.makedirs(PRED_DIR, exist_ok=True)
    MASK_OUT = os.path.join(MAIN_OUTPUT_DIR, "masks");        os.makedirs(MASK_OUT, exist_ok=True)

    if not os.path.exists(FRAME_CSV_PATH):
        pd.DataFrame(columns=FRAME_CSV_COLUMNS).to_csv(FRAME_CSV_PATH, index=False)
    if not os.path.exists(MASK_CSV_PATH):
        pd.DataFrame(columns=MASK_CSV_COLUMNS).to_csv(MASK_CSV_PATH,  index=False)

    frames_skipped_no_masks = 0
    total_frame_rows        = 0
    total_mask_rows         = 0
    errors                  = 0

    for index, row in tqdm(val_list.iterrows(), total=len(val_list),
                           desc="DepthPro Inference + Pre-Mask Loading"):
        if index < START_INDEX:
            continue

        base_fn = (row["folder"] + "_" + row["drive"] + "_"
                   + os.path.splitext(row["filename"])[0])

        try:
            # ── Load image & GT depth ─────────────────────────────────────────
            image_np = np.array(Image.open(row["rgb_path"]).convert("RGB"))
            gt_depth = (np.array(Image.open(row["gt_path"]))
                        .astype(np.uint16) / 256.0)   # 16-bit KITTI → metres
            H, W, _  = image_np.shape
            cx        = W / 2.0

            # ── Load pre-generated masks from disk ────────────────────────────
            mask_dir      = get_mask_dir(row["rgb_path"])
            all_instances = load_masks_for_frame(mask_dir, min_score=MIN_MASK_SCORE)

            if not all_instances:
                frames_skipped_no_masks += 1
                log.debug("No pre-generated masks for %s (dir: %s)", base_fn, mask_dir)
                continue

            # ── DepthPro depth inference (only if masks exist) ────────────────
            pred_raw = run_depthpro(image_np, dp_proc, dp_model)

            # ── Save per-frame arrays (once per frame) ────────────────────────
            rgb_path_out  = os.path.join(RGB_DIR,  f"{base_fn}.png")
            gt_path_out   = os.path.join(GT_DIR,   f"{base_fn}.npy")
            pred_path_out = os.path.join(PRED_DIR, f"{base_fn}_depthpro.npy")
            Image.fromarray(image_np).save(rgb_path_out)
            if SAVE_NPY:
                np.save(gt_path_out,   gt_depth)
                np.save(pred_path_out, pred_raw)

            crop_mask = eigen_crop_mask(gt_depth.shape)

            # ── 1. FRAME-LEVEL EVALUATION ─────────────────────────────────────
            # Evaluate over ALL valid LiDAR pixels in the Eigen crop,
            # completely independently of any object instance masks.
            frame_mask = ((gt_depth > MIN_DEPTH) &
                          (gt_depth < MAX_DEPTH) &
                          crop_mask)
            if np.any(frame_mask):
                f_pts       = int(np.sum(frame_mask))
                f_res       = compute_all_scales(gt_depth, pred_raw, frame_mask)
                f_lidar_avg = float(np.mean(gt_depth[frame_mask]))
                f_lidar_min = float(np.min(gt_depth[frame_mask]))

                frame_row = {
                    "img_id":    index,
                    "file_name": base_fn,
                    "n_pts":     f_pts,
                    **flatten_scale_results(f_res, f_lidar_avg, f_lidar_min),
                    "rgb_image_path":    rgb_path_out,
                    "gt_depth_map_path": gt_path_out   if SAVE_NPY else "",
                    "depthpro_pred_path": pred_path_out if SAVE_NPY else "",
                }
                pd.DataFrame([frame_row]).to_csv(
                    FRAME_CSV_PATH, mode="a", header=False, index=False)
                total_frame_rows += 1

            # ── 2. MASK-LEVEL EVALUATION ──────────────────────────────────────
            # Evaluate independently for each instance mask.
            # final_mask reuses frame_mask to avoid recomputing the GT validity
            # check — object pixels ∩ valid GT ∩ Eigen crop.
            for inst in all_instances:
                seg_mask  = inst["mask"]
                score     = inst["score"]
                super_cat = inst["super_cat"]
                sub_cat   = inst["sub_cat"]
                mask_idx  = inst["local_idx"]
                mask_uid  = inst.get("uid", "")

                if seg_mask.shape != (H, W):
                    log.warning(
                        "Mask shape %s != image shape %s for %s (uid=%s); skipping.",
                        seg_mask.shape, (H, W), base_fn, mask_uid)
                    continue

                # object pixels ∩ valid GT ∩ Eigen crop
                eval_mask = seg_mask & frame_mask
                if not np.any(eval_mask):
                    continue

                m_pts       = int(np.sum(eval_mask))
                m_res       = compute_all_scales(gt_depth, pred_raw, eval_mask)
                m_lidar_avg = float(np.mean(gt_depth[eval_mask]))
                m_lidar_min = float(np.min(gt_depth[eval_mask]))

                _, x_coords = np.where(seg_mask)
                centroid_x  = float(np.mean(x_coords)) if len(x_coords) > 0 else cx
                angle_deg   = math.degrees(
                    math.atan((centroid_x - cx) / FOCAL_LENGTH_X))

                # ── Save binary eval-mask as .npy ─────────────────────────────
                mask_npy_path = ""
                if SAVE_NPY:
                    mask_npy_path = os.path.join(
                        MASK_OUT, f"{base_fn}_mask_{mask_idx:04d}.npy")
                    np.save(mask_npy_path, eval_mask)

                mask_row = {
                    "img_id":           index,
                    "file_name":        base_fn,
                    "mask_id":          mask_idx,
                    "mask_uid":         mask_uid,
                    "mask_score":       score,
                    "super_cat":        super_cat,
                    "sub_cat":          sub_cat,
                    "horizontal_angle": angle_deg,
                    "n_pts":            m_pts,
                    **flatten_scale_results(m_res, m_lidar_avg, m_lidar_min),
                    "rgb_image_path":    rgb_path_out,
                    "gt_depth_map_path": gt_path_out   if SAVE_NPY else "",
                    "depthpro_pred_path": pred_path_out if SAVE_NPY else "",
                    "mask_path":         str(inst["path"]),
                }
                pd.DataFrame([mask_row]).to_csv(
                    MASK_CSV_PATH, mode="a", header=False, index=False)
                total_mask_rows += 1

        except Exception as exc:
            log.error("Error on %s: %s", base_fn, exc, exc_info=True)
            errors += 1
            continue

    log.info("Processing complete.")
    log.info("  Frame CSV → %s  (%d rows)", FRAME_CSV_PATH, total_frame_rows)
    log.info("  Mask  CSV → %s  (%d rows)", MASK_CSV_PATH,  total_mask_rows)
    log.info("  Frames skipped (no masks): %d", frames_skipped_no_masks)
    log.info("  Errors: %d", errors)

print("process_dataset() defined.")

process_dataset() defined.


## 10 · Analysis & Visualisation Helpers

In [11]:
SCALE_COLORS = {
    "unscaled": "#e74c3c",
    "median":   "#27ae60",
    "mean":     "#2980b9",
    "affine":   "#8e44ad",
}
SCALE_LABELS = {
    "unscaled": "Unscaled",
    "median":   "Median Scale",
    "mean":     "Mean Scale",
    "affine":   "Affine Scale",
}


def load_results() -> tuple:
    """Read both CSVs and drop rows with any missing core metric columns.

    Returns
    -------
    (df_frame, df_mask) — two DataFrames, one per CSV.
    """
    req_cols = [
        "unscaled_rmse", "median_rmse", "mean_rmse", "affine_rmse",
        "lidar_avg_depth", "lidar_min_depth",
    ]
    df_frame = pd.read_csv(FRAME_CSV_PATH).dropna(subset=req_cols)
    df_mask  = pd.read_csv(MASK_CSV_PATH).dropna(subset=req_cols)
    return df_frame, df_mask


def true_rmse(df_sub: pd.DataFrame, rmse_col: str) -> float:
    """Pixel-weighted RMSE: sqrt(sum(RMSE_i^2 * n_pts_i) / sum(n_pts_i)).

    Aggregates per-row RMSEs into one overall figure, weighting each row
    by its pixel count so large-pixel regions dominate appropriately.
    """
    valid = df_sub[[rmse_col, "n_pts"]].dropna()
    if valid.empty or valid["n_pts"].sum() == 0:
        return float("nan")
    total_sse = (valid[rmse_col] ** 2 * valid["n_pts"]).sum()
    total_pts = valid["n_pts"].sum()
    return float(np.sqrt(total_sse / total_pts))


def _save_fig(fig: plt.Figure, name: str, dpi: int = 150) -> None:
    os.makedirs(PLOT_DIR, exist_ok=True)
    path = os.path.join(PLOT_DIR, name)
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    log.info("  ✓  %s", name)

print("Analysis helpers defined.")

Analysis helpers defined.


## 11 · Plot Functions

In [12]:
# ── Plot 00: Summary Table ────────────────────────────────────────────────────
def plot_summary_table(df_frame: pd.DataFrame, df_mask: pd.DataFrame) -> None:
    """
    Render all three evaluation tables into a single 00_summary_table.png:
      1. Frame-Level Metrics  (whole-image, pixel-weighted per frame)
      2. Overall Mask-Level Metrics  (all valid masks combined, pixel-weighted)
      3. Per-Category Mask RMSE  (pixel-weighted True RMSE per super-category)
    """
    specs = [
        ("Unscaled",     "unscaled_rmse","unscaled_abs_rel","unscaled_delta1", None,           None),
        ("Median Scale", "median_rmse",  "median_abs_rel",  "median_delta1",   "median_scale", None),
        ("Mean Scale",   "mean_rmse",    "mean_abs_rel",    "mean_delta1",     "mean_scale",   None),
        ("Affine Scale", "affine_rmse",  "affine_abs_rel",  "affine_delta1",   "affine_scale", "affine_shift"),
    ]
    col_labels_main = ["Scale Method", "Avg Scale", "Avg Shift",
                       "True RMSE (m)", "Avg Abs Rel", "Avg \u03b4<1.25"]

    def _make_rows(df):
        rows = []
        for label, rc, ac, dc, sc, shc in specs:
            sv  = df[sc].mean()  if sc  else 1.0
            shv = df[shc].mean() if shc else 0.0
            rows.append([label,
                         f"{sv:.4f}",
                         f"{shv:.4f}" if shc else "\u2014",
                         f"{true_rmse(df, rc):.4f}",
                         f"{df[ac].mean():.4f}",
                         f"{df[dc].mean():.4f}"])
        return rows

    rows_frame = _make_rows(df_frame)
    rows_mask  = _make_rows(df_mask)

    # ── Per-category rows ──────────────────────────────────────────────────────
    cats = sorted(df_mask["super_cat"].dropna().unique())
    col_labels_cat = ["Category", "#Masks", "Unscaled RMSE", "Median RMSE",
                      "Mean RMSE", "Affine RMSE"]
    rows_cat = []
    for cat in cats:
        sub = df_mask[df_mask["super_cat"] == cat]
        rows_cat.append([cat, f"{len(sub):,}",
                         f"{true_rmse(sub, 'unscaled_rmse'):.4f}",
                         f"{true_rmse(sub, 'median_rmse'):.4f}",
                         f"{true_rmse(sub, 'mean_rmse'):.4f}",
                         f"{true_rmse(sub, 'affine_rmse'):.4f}"])

    HDR_COLOR  = "#2c3e50"
    HDR_COLOR2 = "#1a5276"
    HDR_COLOR3 = "#154360"

    def _style_table(tbl, n_rows, n_cols, hdr_color):
        tbl.auto_set_font_size(False); tbl.set_fontsize(10); tbl.scale(1.0, 1.7)
        for j in range(n_cols):
            tbl[0, j].set_facecolor(hdr_color)
            tbl[0, j].set_text_props(color="white", fontweight="bold")
        for i in range(1, n_rows + 1):
            bg = "#f0f4f8" if i % 2 == 1 else "white"
            for j in range(n_cols): tbl[i, j].set_facecolor(bg)

    fig = plt.figure(figsize=(16, 2.0 + 1.8 + 1.8 + 0.5 * max(len(cats), 2)))
    gs  = fig.add_gridspec(3, 1, hspace=0.55,
                           height_ratios=[4, 4, max(len(cats), 2)])
    fig.suptitle(
        "DepthPro (apple/DepthPro-hf) \u2014 Full Evaluation Summary  (Pixel-Weighted True RMSE)\n"
        f"Frames: {len(df_frame):,}   |   Valid Masks: {len(df_mask):,}",
        fontsize=13, fontweight="bold", y=1.01
    )

    ax1 = fig.add_subplot(gs[0]); ax1.axis("off")
    ax1.set_title("Frame-Level Metrics  (whole-image, pixel-weighted per frame)",
                  fontsize=11, fontweight="bold", loc="left", pad=6, color="#2c3e50")
    tbl1 = ax1.table(cellText=rows_frame, colLabels=col_labels_main,
                     cellLoc="center", loc="center", bbox=[0, 0, 1, 1])
    _style_table(tbl1, len(rows_frame), len(col_labels_main), HDR_COLOR)

    ax2 = fig.add_subplot(gs[1]); ax2.axis("off")
    ax2.set_title("Overall Mask-Level Metrics  (all valid masks combined, pixel-weighted)",
                  fontsize=11, fontweight="bold", loc="left", pad=6, color="#1a5276")
    tbl2 = ax2.table(cellText=rows_mask, colLabels=col_labels_main,
                     cellLoc="center", loc="center", bbox=[0, 0, 1, 1])
    _style_table(tbl2, len(rows_mask), len(col_labels_main), HDR_COLOR2)

    ax3 = fig.add_subplot(gs[2]); ax3.axis("off")
    ax3.set_title("Per-Category Mask RMSE  (pixel-weighted True RMSE per super-category)",
                  fontsize=11, fontweight="bold", loc="left", pad=6, color="#154360")
    tbl3 = ax3.table(cellText=rows_cat, colLabels=col_labels_cat,
                     cellLoc="center", loc="center", bbox=[0, 0, 1, 1])
    _style_table(tbl3, len(rows_cat), len(col_labels_cat), HDR_COLOR3)

    plt.tight_layout()
    _save_fig(fig, "00_summary_table.png", dpi=180)


# ── Plot 01: GT vs Predicted Depth ────────────────────────────────────────────
def plot_gt_vs_pred(df: pd.DataFrame) -> None:
    variants = [("Unscaled","unscaled_avg_depth"),("Median Scale","median_avg_depth"),
                ("Mean Scale","mean_avg_depth"),("Affine Scale","affine_avg_depth")]
    fig, axes = plt.subplots(2, 2, figsize=(16, 14)); axes = axes.flatten()
    for ax, (title, ycol), color in zip(axes, variants, SCALE_COLORS.values()):
        sns.regplot(ax=ax, data=df, x="lidar_avg_depth", y=ycol,
                    scatter_kws={"s": 25, "alpha": 0.35, "color": color},
                    line_kws={"color": "red", "linewidth": 2}, lowess=True)
        ax.plot([0, MAX_DEPTH], [0, MAX_DEPTH], "k--", lw=1.5, label="Ideal 1:1")
        ax.set_title(f"DepthPro  |  {title}", fontsize=12)
        ax.set_xlabel("LiDAR GT Depth (m)"); ax.set_ylabel("Predicted Depth (m)")
        ax.set_xlim(0, MAX_DEPTH); ax.set_ylim(0, MAX_DEPTH); ax.legend(fontsize=9)
    fig.suptitle("DepthPro — GT Depth vs Predicted Depth (all scale variants)",
                 fontsize=15, y=1.01)
    plt.tight_layout(); _save_fig(fig, "01_gt_vs_pred_all_scales.png")


# ── Plot 02: GT Depth vs RMSE ─────────────────────────────────────────────────
def plot_gt_vs_rmse(df: pd.DataFrame) -> None:
    variants = [("Unscaled","unscaled_rmse"),("Median Scale","median_rmse"),
                ("Mean Scale","mean_rmse"),("Affine Scale","affine_rmse")]
    ymax = df[[v[1] for v in variants]].max().max() * 1.12
    fig, axes = plt.subplots(2, 2, figsize=(16, 14)); axes = axes.flatten()
    for ax, (title, ycol), color in zip(axes, variants, SCALE_COLORS.values()):
        sns.regplot(ax=ax, data=df, x="lidar_avg_depth", y=ycol,
                    scatter_kws={"s": 25, "alpha": 0.35, "color": color},
                    line_kws={"color": "red", "linewidth": 2}, lowess=True)
        ax.set_title(f"DepthPro  |  {title}", fontsize=12)
        ax.set_xlabel("LiDAR GT Depth (m)"); ax.set_ylabel("RMSE (m)")
        ax.set_xlim(0, MAX_DEPTH); ax.set_ylim(0, ymax)
    fig.suptitle("DepthPro — GT Depth vs RMSE (all scale variants)",
                 fontsize=15, y=1.01)
    plt.tight_layout(); _save_fig(fig, "02_gt_vs_rmse_all_scales.png")


# ── Plot 03: Horizontal Angle vs RMSE ────────────────────────────────────────
def plot_angle_vs_rmse(df: pd.DataFrame) -> None:
    variants = [("Unscaled","unscaled_rmse"),("Median Scale","median_rmse"),
                ("Mean Scale","mean_rmse"),("Affine Scale","affine_rmse")]
    ymax = df[[v[1] for v in variants]].max().max() * 1.12
    fig, axes = plt.subplots(2, 2, figsize=(16, 14)); axes = axes.flatten()
    for ax, (title, ycol), color in zip(axes, variants, SCALE_COLORS.values()):
        sns.regplot(ax=ax, data=df, x="horizontal_angle", y=ycol,
                    scatter_kws={"s": 25, "alpha": 0.35, "color": color},
                    line_kws={"color": "red", "linewidth": 2}, lowess=True)
        ax.set_title(f"DepthPro  |  {title}: Angle vs RMSE", fontsize=12)
        ax.set_xlabel("Horizontal Angle (°)"); ax.set_ylabel("RMSE (m)")
        ax.set_ylim(0, ymax)
    fig.suptitle("DepthPro — Horizontal Angle vs RMSE (all scale variants)",
                 fontsize=15, y=1.01)
    plt.tight_layout(); _save_fig(fig, "03_angle_vs_rmse_all_scales.png")


# ── Plot 04: Scale Method Comparison Bar Chart ────────────────────────────────
def plot_scale_comparison_bars(df: pd.DataFrame) -> None:
    metrics_spec = [
        ("RMSE (m)  \u2193",["unscaled_rmse","median_rmse","mean_rmse","affine_rmse"]),
        ("Abs Rel   \u2193",["unscaled_abs_rel","median_abs_rel","mean_abs_rel","affine_abs_rel"]),
        ("\u03b4 < 1.25  \u2191",["unscaled_delta1","median_delta1","mean_delta1","affine_delta1"]),
    ]
    labels = ["Unscaled","Median","Mean","Affine"]
    colors = list(SCALE_COLORS.values())
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    for ax, (metric_name, cols) in zip(axes, metrics_spec):
        # Use pixel-weighted True RMSE for RMSE columns; simple mean for others
        if "rmse" in cols[0].lower():
            vals = [true_rmse(df, c) for c in cols]
        else:
            vals = [df[c].mean() if c in df.columns else 0.0 for c in cols]
        bars = ax.bar(labels, vals, color=colors, edgecolor="white", width=0.6)
        ax.bar_label(bars, fmt="%.4f", padding=4, fontsize=10)
        ax.set_title(metric_name, fontsize=13); ax.set_ylabel(metric_name)
        ax.set_ylim(0, max(vals) * 1.25 if max(vals) > 0 else 1)
        ax.grid(axis="y", alpha=0.35)
    fig.suptitle("DepthPro — Scale Method Comparison  (Pixel-Weighted True RMSE)",
                 fontsize=15, y=1.02)
    plt.tight_layout(); _save_fig(fig, "04_scale_comparison_bars.png")


# ── Plot 05: Model vs LiDAR by Object Category ───────────────────────────────
def plot_model_vs_lidar_by_category(df: pd.DataFrame) -> None:
    cats = sorted(df["super_cat"].dropna().unique())
    n = len(cats)
    if n == 0: return
    fig, axes = plt.subplots(1, max(n, 1), figsize=(7 * n, 6))
    if n == 1: axes = [axes]
    for ax, cat in zip(axes, cats):
        sub = df[df["super_cat"] == cat]
        for key in ["unscaled","median","mean","affine"]:
            ax.scatter(sub["lidar_avg_depth"], sub[f"{key}_avg_depth"],
                       s=18, alpha=0.4, c=SCALE_COLORS[key], label=SCALE_LABELS[key])
        ax.plot([0, MAX_DEPTH], [0, MAX_DEPTH], "k--", lw=1.5, label="Ideal 1:1")
        ax.set_xlim(0, MAX_DEPTH); ax.set_ylim(0, MAX_DEPTH)
        ax.set_xlabel("LiDAR GT Depth (m)"); ax.set_ylabel("Predicted Depth (m)")
        ax.set_title(f"Category: {cat}", fontsize=12); ax.legend(fontsize=8)
    fig.suptitle("DepthPro — Predicted Depth vs LiDAR GT (by Object Category)",
                 fontsize=14)
    plt.tight_layout(); _save_fig(fig, "05_model_vs_lidar_by_category.png")


# ── Plot 06: RMSE Distribution Histograms ─────────────────────────────────────
def plot_rmse_distributions(df: pd.DataFrame) -> None:
    variants = [("Unscaled","unscaled_rmse"),("Median Scale","median_rmse"),
                ("Mean Scale","mean_rmse"),("Affine Scale","affine_rmse")]
    fig, axes = plt.subplots(2, 2, figsize=(16, 10)); axes = axes.flatten()
    for ax, (label, col), color in zip(axes, variants, SCALE_COLORS.values()):
        vals = df[col].dropna()
        ax.hist(vals, bins=60, color=color, alpha=0.72, edgecolor="white")
        ax.axvline(vals.mean(), color="black", ls="--", lw=1.8,
                   label=f"Mean   = {vals.mean():.3f} m")
        ax.axvline(vals.median(), color="grey", ls=":", lw=1.8,
                   label=f"Median = {vals.median():.3f} m")
        ax.set_title(f"RMSE Distribution — {label}", fontsize=12)
        ax.set_xlabel("RMSE (m)"); ax.set_ylabel("Count"); ax.legend(fontsize=9)
    fig.suptitle("DepthPro — RMSE Distributions Across All Masks", fontsize=14)
    plt.tight_layout(); _save_fig(fig, "06_rmse_distributions.png")


# ── Plot 07: Scale Factor Distributions ──────────────────────────────────────
def plot_scale_distributions(df: pd.DataFrame) -> None:
    spec = [("median_scale","Median Scale",SCALE_COLORS["median"]),
            ("mean_scale",  "Mean Scale",  SCALE_COLORS["mean"]),
            ("affine_scale","Affine Scale",SCALE_COLORS["affine"])]
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, (col, label, color) in zip(axes, spec):
        vals = df[col].dropna()
        ax.hist(vals, bins=60, color=color, alpha=0.72, edgecolor="white")
        ax.axvline(1.0, color="black", ls="--", lw=1.8, label="Scale = 1.0")
        ax.axvline(vals.mean(), color="red", ls=":", lw=1.8,
                   label=f"Mean = {vals.mean():.3f}")
        ax.set_title(f"{label} Distribution", fontsize=12)
        ax.set_xlabel("Scale Factor"); ax.set_ylabel("Count"); ax.legend(fontsize=9)
    fig.suptitle("DepthPro — Scale Factor Distributions", fontsize=14)
    plt.tight_layout(); _save_fig(fig, "07_scale_distributions.png")


# ── Plot 08: 3D Error Surfaces ────────────────────────────────────────────────
def plot_3d_error_surfaces(df: pd.DataFrame, degree: int = 2) -> None:
    variants = [("Unscaled","unscaled_rmse","#e74c3c"),
                ("Median Scale","median_rmse","#27ae60"),
                ("Mean Scale","mean_rmse","#2980b9"),
                ("Affine Scale","affine_rmse","#8e44ad")]
    xcol, ycol = "lidar_avg_depth", "horizontal_angle"
    df_c = df.dropna(subset=[xcol, ycol] + [v[1] for v in variants])
    if len(df_c) < 10: return
    x_min, x_max = df_c[xcol].min(), df_c[xcol].max()
    y_min, y_max = df_c[ycol].min(), df_c[ycol].max()
    z_min = df_c[[v[1] for v in variants]].min().min()
    z_max = df_c[[v[1] for v in variants]].max().max()
    buf = lambda r: r * 0.08
    fig = plt.figure(figsize=(26, 6))
    fig.suptitle(f"DepthPro — 3D Error Surfaces (poly deg={degree})", fontsize=14)
    for sp_idx, (label, zcol, color) in enumerate(variants, start=1):
        ax = fig.add_subplot(1, 4, sp_idx, projection="3d")
        ax.scatter(df_c[xcol], df_c[ycol], df_c[zcol], c=color, alpha=0.35, s=8)
        Xf = df_c[[xcol, ycol]].values; Zf = df_c[zcol].values
        pipe = make_pipeline(PolynomialFeatures(degree), LinearRegression())
        pipe.fit(Xf, Zf)
        xs, ys = np.meshgrid(np.linspace(x_min, x_max, 40),
                              np.linspace(y_min, y_max, 40))
        zs = pipe.predict(np.c_[xs.ravel(), ys.ravel()]).reshape(xs.shape)
        ax.plot_surface(xs, ys, zs, color=color, alpha=0.22, rstride=1, cstride=1)
        ax.set_xlabel("GT Depth (m)", fontsize=8); ax.set_ylabel("H. Angle (°)", fontsize=8)
        ax.set_zlabel("RMSE (m)", fontsize=8); ax.set_title(label, fontsize=11)
        bx, by, bz = buf(x_max-x_min), buf(y_max-y_min), buf(z_max-z_min)
        ax.set_xlim(x_min-bx, x_max+bx); ax.set_ylim(y_min-by, y_max+by)
        ax.set_zlim(z_min-bz, z_max+bz)
    plt.tight_layout(); _save_fig(fig, "08_3d_error_surfaces.png")


# ── Plot 09: Depth-Bin Bias Analysis ─────────────────────────────────────────
def plot_depth_bin_analysis(df: pd.DataFrame) -> None:
    bins   = np.arange(0, MAX_DEPTH + 10, 10)
    labels = [f"{int(b)}\u2013{int(b+10)}m" for b in bins[:-1]]
    df2    = df.copy()
    df2["depth_bin"] = pd.cut(df2["lidar_avg_depth"], bins=bins, labels=labels)
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    ax = axes[0]
    for key in ["unscaled","median","mean","affine"]:
        agg = df2.groupby("depth_bin", observed=True)[f"{key}_rmse"].mean()
        ax.plot(agg.index, agg.values, "o-", color=SCALE_COLORS[key],
                label=SCALE_LABELS[key], linewidth=2)
    ax.set_xlabel("Depth Bin"); ax.set_ylabel("Mean RMSE (m)")
    ax.set_title("RMSE by Depth Bin — Scale Comparison")
    ax.legend(); ax.tick_params(axis="x", rotation=45); ax.grid(alpha=0.3)
    ax = axes[1]
    for key in ["unscaled","median","mean","affine"]:
        agg_pred  = df2.groupby("depth_bin", observed=True)[f"{key}_avg_depth"].mean()
        agg_lidar = df2.groupby("depth_bin", observed=True)["lidar_avg_depth"].mean()
        bias = agg_pred - agg_lidar
        ax.plot(bias.index, bias.values, "o-", color=SCALE_COLORS[key],
                label=SCALE_LABELS[key], linewidth=2)
    ax.axhline(0, color="black", ls="--", lw=1.5, label="Zero bias")
    ax.set_xlabel("Depth Bin"); ax.set_ylabel("Prediction Bias (m)")
    ax.set_title("Depth Bias by Bin (Pred \u2212 LiDAR)")
    ax.legend(); ax.tick_params(axis="x", rotation=45); ax.grid(alpha=0.3)
    fig.suptitle("DepthPro — Depth-Bin Analysis", fontsize=14)
    plt.tight_layout(); _save_fig(fig, "09_depth_bin_analysis.png")


# ── Plot 10: RMSE per Object Category ────────────────────────────────────────
def plot_rmse_by_category(df: pd.DataFrame) -> None:
    cats = sorted(df["super_cat"].dropna().unique())
    if len(cats) < 2: return
    keys, n_keys = ["unscaled","median","mean","affine"], 4
    x, w = np.arange(len(cats)), 0.18
    fig, ax = plt.subplots(figsize=(13, 6))
    for i, key in enumerate(keys):
        vals   = [true_rmse(df[df["super_cat"] == c], f"{key}_rmse") for c in cats]
        offset = (i - (n_keys - 1) / 2.0) * w
        bars   = ax.bar(x + offset, vals, width=w,
                        label=SCALE_LABELS[key], color=SCALE_COLORS[key])
        ax.bar_label(bars, fmt="%.2f", fontsize=8, padding=2)
    ax.set_xticks(x); ax.set_xticklabels(cats, fontsize=11)
    ax.set_ylabel("Pixel-Weighted True RMSE (m)")
    ax.set_title("DepthPro — RMSE per Object Category  (Pixel-Weighted)", fontsize=13)
    ax.legend(); ax.grid(axis="y", alpha=0.35)
    plt.tight_layout(); _save_fig(fig, "10_rmse_by_category.png")


# ── Plot 11: Scale Factor vs GT Depth ────────────────────────────────────────
def plot_scale_vs_depth(df: pd.DataFrame) -> None:
    spec = [("median_scale","Median Scale",SCALE_COLORS["median"]),
            ("mean_scale",  "Mean Scale",  SCALE_COLORS["mean"]),
            ("affine_scale","Affine Scale",SCALE_COLORS["affine"])]
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, (col, label, color) in zip(axes, spec):
        sns.regplot(ax=ax, data=df, x="lidar_avg_depth", y=col,
                    scatter_kws={"s": 18, "alpha": 0.30, "color": color},
                    line_kws={"color": "red", "linewidth": 2}, lowess=True)
        ax.axhline(1.0, color="black", ls="--", lw=1.5, label="Scale = 1.0")
        ax.set_xlabel("LiDAR GT Depth (m)"); ax.set_ylabel("Scale Factor")
        ax.set_title(f"{label} vs Depth"); ax.legend(fontsize=9)
    fig.suptitle("DepthPro — Scale Factor vs Ground-Truth Depth", fontsize=14)
    plt.tight_layout(); _save_fig(fig, "11_scale_vs_depth.png")


def generate_all_plots(df_frame: pd.DataFrame, df_mask: pd.DataFrame) -> None:
    """Generate and save all 12 analysis plots."""    
    sns.set_style("whitegrid"); sns.set_context("paper", font_scale=1.1)
    log.info("Generating analysis plots \u2192 %s", PLOT_DIR)
    plot_summary_table(df_frame, df_mask)
    plot_gt_vs_pred(df_mask);              plot_gt_vs_rmse(df_mask)
    plot_angle_vs_rmse(df_mask);           plot_scale_comparison_bars(df_mask)
    plot_model_vs_lidar_by_category(df_mask); plot_rmse_distributions(df_mask)
    plot_scale_distributions(df_mask);    plot_3d_error_surfaces(df_mask)
    plot_depth_bin_analysis(df_mask);     plot_rmse_by_category(df_mask)
    plot_scale_vs_depth(df_mask)
    log.info("All 12 plots saved to %s", PLOT_DIR)

print("All plot functions defined.")

All plot functions defined.


## 12 · Terminal Summary Printer

In [13]:
def print_summary(df_frame: pd.DataFrame, df_mask: pd.DataFrame) -> None:
    """
    Print a three-section terminal summary that cleanly separates:
      1. Frame-Level  — whole-image accuracy (all valid LiDAR pixels, Eigen crop)
      2. Overall Mask — accuracy over all valid object-mask pixels combined
      3. Per-Category — accuracy broken down by super-category
    All RMSE figures are pixel-weighted True RMSE.
    """
    W   = 115
    SEP = "\u2550" * W
    sep = "\u2500" * W

    print("\n" + SEP)
    print("  DepthPro (apple/DepthPro-hf) \u2014 Evaluation Summary  (Pixel-Weighted True RMSE)")
    print(f"  Frames evaluated: {len(df_frame):,}   |   Valid Masks: {len(df_mask):,}")
    print(SEP)

    specs = [
        ("Unscaled",     "unscaled_rmse","unscaled_abs_rel","unscaled_delta1", None,           None),
        ("Median Scale", "median_rmse",  "median_abs_rel",  "median_delta1",   "median_scale", None),
        ("Mean Scale",   "mean_rmse",    "mean_abs_rel",    "mean_delta1",     "mean_scale",   None),
        ("Affine Scale", "affine_rmse",  "affine_abs_rel",  "affine_delta1",   "affine_scale", "affine_shift"),
    ]
    hdr = (f"  {'Scale Method':<24}"
           f"{'Avg Scale':>12}{'Avg Shift':>12}"
           f"{'True RMSE (m)':>16}{'Avg Abs Rel':>14}{'Avg Delta <1.25':>12}")

    # ── 1. Frame-Level Metrics ────────────────────────────────────────────────
    print("\n[FRAME-LEVEL METRICS  (Whole-image evaluation, pixel-weighted per frame)]")
    print(hdr); print(sep)
    for label, rc, ac, dc, sc, shc in specs:
        sv  = df_frame[sc].mean()  if sc  else 1.0
        shv = df_frame[shc].mean() if shc else 0.0
        print(
            f"  {label:<24}"
            f"{sv:>12.4f}{shv:>12.4f}"
            f"{true_rmse(df_frame, rc):>16.4f}"
            f"{df_frame[ac].mean():>14.4f}"
            f"{df_frame[dc].mean():>12.4f}"
        )

    # ── 2. Overall Mask-Level Metrics ─────────────────────────────────────────
    print("\n[OVERALL MASK-LEVEL METRICS  (All valid masks combined, pixel-weighted)]")
    print(hdr); print(sep)
    for label, rc, ac, dc, sc, shc in specs:
        sv  = df_mask[sc].mean()  if sc  else 1.0
        shv = df_mask[shc].mean() if shc else 0.0
        print(
            f"  {label:<24}"
            f"{sv:>12.4f}{shv:>12.4f}"
            f"{true_rmse(df_mask, rc):>16.4f}"
            f"{df_mask[ac].mean():>14.4f}"
            f"{df_mask[dc].mean():>12.4f}"
        )

    # ── 3. Per-Category Mask-Level Metrics ────────────────────────────────────
    print("\n[INDIVIDUAL CATEGORY METRICS  (Per super-category, pixel-weighted)]")
    cats = sorted(df_mask["super_cat"].dropna().unique())
    hdr3 = (f"  {'Category':<28}  {'#Masks':>8}"
            f"  {'Unscaled':>12}  {'Median':>12}"
            f"  {'Mean':>12}  {'Affine':>12}")
    print(hdr3); print(sep)
    for cat in cats:
        sub = df_mask[df_mask["super_cat"] == cat]
        print(
            f"  {cat:<28}  {len(sub):>8,}"
            f"  {true_rmse(sub, 'unscaled_rmse'):>12.4f}"
            f"  {true_rmse(sub, 'median_rmse'):>12.4f}"
            f"  {true_rmse(sub, 'mean_rmse'):>12.4f}"
            f"  {true_rmse(sub, 'affine_rmse'):>12.4f}"
        )
    print(SEP + "\n")

    print("  Scale Factor Statistics")
    print(f"  {'Metric':<22}  {'Mean':>9}  {'Median':>9}  {'Std':>9}")
    print("  " + sep[:55])
    for label, col in [
        ("Median Scale",     "median_scale"),
        ("Mean Scale",       "mean_scale"),
        ("Affine Scale",     "affine_scale"),
        ("Affine Shift (m)", "affine_shift"),
    ]:
        if col not in df_mask.columns: continue
        s = df_mask[col].dropna()
        print(f"  {label:<22}  {s.mean():>9.4f}  {s.median():>9.4f}  {s.std():>9.4f}")
    print()

print("print_summary() defined.")

print_summary() defined.


## 13 · Run Inference  *(skip if CSVs already exist)*

Set `SKIP_INFERENCE = True` to jump straight to analysis from existing CSVs.

In [14]:
SKIP_INFERENCE = True

if not SKIP_INFERENCE:
    log.info("\u2500\u2500 STEP 1/3: Indexing KITTI Eigen Split dataset \u2026")
    val_list = index_kitti(use_train=USE_TRAIN)

    log.info("\u2500\u2500 STEP 2/3: Loading DepthPro model \u2026")
    dp_proc, dp_model = load_depthpro()
    # No SAM3 model to load \u2014 masks are read directly from MASK_ROOT

    log.info("\u2500\u2500 STEP 3/3: Processing dataset (masks from %s) \u2026", MASK_ROOT)
    process_dataset(dp_proc, dp_model, val_list)
else:
    log.info("Skipping inference \u2014 will use existing CSVs at:\n  %s\n  %s",
             FRAME_CSV_PATH, MASK_CSV_PATH)

09:25:32  INFO      Skipping inference — will use existing CSVs at:
  /home/arka/Downloads/MMDE/outputs/model_analysis_output/depthpro_premasked_outputs_FINAL/depthpro_frame_results.csv
  /home/arka/Downloads/MMDE/outputs/model_analysis_output/depthpro_premasked_outputs_FINAL/depthpro_mask_results.csv


## 14 · Analysis & Plots

In [15]:
# Set SKIP_PLOTS = True to only print summary without regenerating plots
SKIP_PLOTS = False

if os.path.exists(FRAME_CSV_PATH) and os.path.exists(MASK_CSV_PATH):
    log.info("Loading results CSVs \u2026")
    df_frame, df_mask = load_results()
    print_summary(df_frame, df_mask)

    if not SKIP_PLOTS:
        generate_all_plots(df_frame, df_mask)
else:
    log.warning(
        "Results CSVs not found at:\n  %s\n  %s\nRun the inference cell first.",
        FRAME_CSV_PATH, MASK_CSV_PATH,
    )

09:25:32  INFO      Loading results CSVs …
09:25:33  INFO      Generating analysis plots → /home/arka/Downloads/MMDE/outputs/model_analysis_output/depthpro_premasked_outputs_FINAL/plots



═══════════════════════════════════════════════════════════════════════════════════════════════════════════════════
  DepthPro (apple/DepthPro-hf) — Evaluation Summary  (Pixel-Weighted True RMSE)
  Frames evaluated: 652   |   Valid Masks: 12,808
═══════════════════════════════════════════════════════════════════════════════════════════════════════════════════

[FRAME-LEVEL METRICS  (Whole-image evaluation, pixel-weighted per frame)]
  Scale Method               Avg Scale   Avg Shift   True RMSE (m)   Avg Abs RelAvg Delta <1.25
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  Unscaled                      1.0000      0.0000          3.7375        0.1446      0.8276
  Median Scale                  0.9448      0.0000          3.0477        0.0647      0.9573
  Mean Scale                    0.9747      0.0000          2.7451        0.0718      0.9619
  Affine Scale                  1.0450     -1.0405          2.5362     

09:25:34  INFO        ✓  00_summary_table.png
09:26:01  INFO        ✓  01_gt_vs_pred_all_scales.png
09:26:29  INFO        ✓  02_gt_vs_rmse_all_scales.png
09:26:56  INFO        ✓  03_angle_vs_rmse_all_scales.png
09:26:56  INFO        ✓  04_scale_comparison_bars.png
09:26:59  INFO        ✓  05_model_vs_lidar_by_category.png
09:27:00  INFO        ✓  06_rmse_distributions.png
09:27:01  INFO        ✓  07_scale_distributions.png
09:27:04  INFO        ✓  08_3d_error_surfaces.png
09:27:05  INFO        ✓  09_depth_bin_analysis.png
09:27:06  INFO        ✓  10_rmse_by_category.png
09:27:26  INFO        ✓  11_scale_vs_depth.png
09:27:26  INFO      All 12 plots saved to /home/arka/Downloads/MMDE/outputs/model_analysis_output/depthpro_premasked_outputs_FINAL/plots
